# Team 5 & 6 — Machine Learning Benchmarking, Tuning & Evaluation

This notebook demonstrates automated machine learning model benchmarking, cross-validation, hyperparameter tuning, and comprehensive metrics evaluation with **AutoAnalyst AI**.

In [ ]:
from pathlib import Path

import pandas as pd

from autoanalyst.data_loading.loader import load_dataset
from autoanalyst.evaluation.evaluator import evaluate_classification
from autoanalyst.feature_engineering.feature_builder import encode_categorical_columns
from autoanalyst.modeling.classification import benchmark_classification_models
from autoanalyst.preprocessing.cleaner import clean_dataset

dataset_path = (
    Path("../data/sample/example.csv")
    if Path("../data/sample/example.csv").exists()
    else Path("data/sample/example.csv")
)
raw_df = load_dataset(dataset_path)
clean_res = clean_dataset(raw_df, missing_strategy="median")
cleaned_df = clean_res.cleaned_df

# Prepare model features
target_col = "loan_status"
cat_cols = [c for c in cleaned_df.select_dtypes(include=["object", "string"]).columns if c != target_col]
model_df = encode_categorical_columns(cleaned_df, columns=cat_cols)

X = model_df.drop(columns=[target_col])
y = model_df[target_col]
print(f"Prepared features: {X.shape[1]} columns, {X.shape[0]} rows")

## 1. Automated Model Zoo Benchmark

In [ ]:
leaderboard, champion = benchmark_classification_models(X, y, cv=3, metric="f1_macro")
print("=== Model Leaderboard ===")
leaderboard

## 2. Champion Training & Comprehensive Evaluation

In [ ]:
X_train, X_test, y_train, y_test = champion.train(X, y, test_size=0.3)
y_pred = champion.predict(X_test)
y_proba = champion.predict_proba(X_test)

eval_results = evaluate_classification(y_test, y_pred, y_proba=y_proba)
print(f"Accuracy: {eval_results['accuracy']:.4f}")
print(f"Macro F1: {eval_results['f1_macro']:.4f}")
print(f"ROC-AUC:  {eval_results.get('roc_auc')}")
print("\nConfusion Matrix:")
print(pd.DataFrame(eval_results["confusion_matrix"]))